In [1]:
import os
import math
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import pynwb
import h5py

from sklearn.decomposition import PCA

In [2]:
def get_continuous_behaviors(file_handle):
	try:
		data = pd.DataFrame({key: file_handle[f'/processing/Behavior/continuous_behaviors/{key}/data'][:] for key in file_handle[f'/processing/Behavior/continuous_behaviors'].keys()})
	except KeyError:
		print(f'No continuous behavior data for {os.path.basename(file_handle.filename)}')
		data = None
	return data

def get_discrete_states(file_handle):
	try:
		data = pd.DataFrame({key: file_handle[f'/processing/Behavior/discrete_states/{key}/data'][:] for key in file_handle[f'/processing/Behavior/discrete_states'].keys()})
	except KeyError:
		print(f'No discrete state data for {os.path.basename(file_handle.filename)}')
		data = None
	return data

def get_eigenworms(file_handle):
	try:
		data = pd.DataFrame({key: file_handle[f'/processing/Behavior/eigenworms/{key}/data'][:] for key in file_handle[f'/processing/Behavior/eigenworms'].keys()})
	except KeyError:
		print(f'No eigenworm data for {os.path.basename(file_handle.filename)}')
		data = None
	return data

def get_kymograph(file_handle):
	try:
		data = pd.DataFrame({key: file_handle[f'/processing/Behavior/kymograph/{key}/data'][:] for key in file_handle[f'/processing/Behavior/kymograph'].keys()})
	except KeyError:
		print(f'No kymograph data for {os.path.basename(file_handle.filename)}')
		data = None
	return data

def get_neuronal_traces(file_handle):
	try:
		data = pd.DataFrame(file_handle[f'/processing/CalciumActivity/SignalDFoF/SignalCalciumImResponseSeries/data'][:],
							 columns=[n_id.decode('utf-8') for n_id in file_handle[f'/processing/CalciumActivity/NeuronIDs/labels'][:]])
	except KeyError:
		print(f'No neuronal trace data for {os.path.basename(file_handle.filename)}')
		data = None
	return data


prefix = f'{os.getcwd()}/../data/{os.path.basename(os.getcwd())}/'
path = prefix + 'exported_data_gcamp/'
hdf5files = [h5py.File(path + filename, 'r') for filename in os.listdir(path)]

In [3]:
continuous_behaviors = [get_continuous_behaviors(file) for file in hdf5files]
discrete_states = [get_discrete_states(file) for file in hdf5files]
eigenworms = [get_eigenworms(file) for file in hdf5files]
kymographs = [get_kymograph(file) for file in hdf5files]
neuronal_traces = [get_neuronal_traces(file) for file in hdf5files]

No continuous behavior data for 2024-01-17_14-05_wt_FM_worm5-2024-01-17_no_image_data.nwb
No continuous behavior data for 2023-11-30_14-31_wt_worm5_FM-2023-11-30_no_image_data.nwb
No continuous behavior data for 2023-12-19_17-35_wt_FM_worm4-2023-12-19_no_image_data.nwb
No continuous behavior data for 2023-12-19_18-27_wt_FM_worm5again2-2023-12-19_no_image_data.nwb
No continuous behavior data for 2023-11-30_13-58_wt_worm3_FM-2023-11-30_no_image_data.nwb
No discrete state data for 2024-01-17_14-05_wt_FM_worm5-2024-01-17_no_image_data.nwb
No discrete state data for 2023-11-30_14-31_wt_worm5_FM-2023-11-30_no_image_data.nwb
No discrete state data for 2023-12-19_17-35_wt_FM_worm4-2023-12-19_no_image_data.nwb
No discrete state data for 2023-12-19_18-27_wt_FM_worm5again2-2023-12-19_no_image_data.nwb
No discrete state data for 2023-11-30_13-58_wt_worm3_FM-2023-11-30_no_image_data.nwb
No eigenworm data for 2024-01-17_14-05_wt_FM_worm5-2024-01-17_no_image_data.nwb
No eigenworm data for 2023-11-30_

In [4]:
neuronal_modes = list()
neuronal_residuals = list()
for trace in neuronal_traces:
	signal = trace.values
	pca = PCA(n_components=2)
	signal_reduced = pca.fit_transform(signal)
	signal_reconstructed = pca.inverse_transform(signal_reduced)
	signal_residual = signal - signal_reconstructed
	neuronal_modes.append(pd.DataFrame(data=signal_reduced, columns=['PC1', 'PC2']))
	neuronal_residuals.append(pd.DataFrame(data=signal_reconstructed, columns=trace.columns))

In [5]:
unique_labels = sorted(set([label for signal in neuronal_traces for label in signal.columns if label != '']))

In [7]:
data = {neuron: (np.empty((0, 5)), np.empty((0, 2)), np.empty((0, 1))) for neuron in unique_labels}	# X, X_man, Y
for label in unique_labels:
	if label != '':
		print(label)
		for ews, modes, residuals in zip(eigenworms, neuronal_modes, neuronal_residuals):
			if label in residuals.columns and ews is not None:
				Y_new = residuals[label].values.reshape(-1, 1)
				X_prev, X_man_prev, Y_prev = data[label]
				X = np.vstack((X_prev, ews))
				X_man = np.vstack((X_man_prev, modes))
				Y = np.vstack((Y_prev, Y_new))
				data[label] = X, X_man, Y

AIBL
AIBR
ALA
ANTIcor2R?
ANTIcorL
ANTIcorR
AQR
AQR_
AUAL
AUAL'
AUAR
AUAR'
AUAR_
AVAL
AVAL_
AVAR
AVBL
AVBL_
AVBR
AVEL
AVEL_
AVER
AVER_
AVFL
AVFR
AWAL
AWAR
AWBL
AWBR
AWBR_
BAGL
BAGR
CEPDR
DA01
DA01_
DB01
DB02
DB02_
DD01
IL11LR
IL1L
IL1R
IL1VL
IL2
IL2DL
IL2DR
IL2L
IL2R
IL2V
IL2VR
OLQDL
OLQDR
OLQVL
OLQVR
RAIL
RIAL
RIAR
RIB/AUA
RIBL
RIBL_
RIBR
RIBR_
RID
RIM/AUAR_
RIML
RIML_
RIMR
RIS
RIVL
RIVR
RMDDL
RMDDR
RMDDR_
RMDVL
RMDVL_
RMDVR
RMED
RMEL
RMER
RMEV
SAAVL
SAAVR
SIADL
SIADR
SIADR_
SIAVL
SIAVR
SIAVR_
SMDDL
SMDDR
SMDVL
SMDVL_
SMDVR
URADL
URADR
URVR
URXL
URXR
URYDL
URYDR
URYVL
URYVR
VA01
VA01_
VA02
VA02_
VB01
VB01_
VB02
VB03
VG_anter_FWD_no_curve_L
VG_middle_ramping_L
VG_middle_ramping_R
VG_post_FWD_L
VG_post_FWD_R


In [8]:
print(data)

{'AIBL': (array([[-0.65963924,  1.28204862,  1.57326062, -0.0854296 ,  0.38124442],
       [-0.78052968,  1.02043071,  1.4952356 , -0.54511168,  0.23718497],
       [-0.95569924,  0.56574898,  1.30253398, -1.26718152, -0.01291362],
       ...,
       [-1.0993804 ,  1.57294016,  0.39364356, -2.87684782,  2.96428577],
       [-1.37123855,  1.38437254, -0.05998126, -2.89629156,  2.90964451],
       [-1.54803826,  1.2654421 , -0.34470176, -2.88580315,  2.88560627]],
      shape=(6668, 5)), array([[-0.44632847, -0.30806931],
       [-0.70091603, -0.29449952],
       [-0.85533847, -0.25349734],
       ...,
       [ 1.07898781, -0.04963058],
       [ 1.07792401, -0.05024585],
       [ 1.07470679, -0.0506565 ]], shape=(6668, 2)), array([[-0.01084676],
       [-0.03855206],
       [-0.05659235],
       ...,
       [ 0.18755348],
       [ 0.18742277],
       [ 0.18699386]], shape=(6668, 1))), 'AIBR': (array([[ 0.74756513, -0.41065659,  2.05508087, -0.52068379, -0.43020409],
       [ 0.6120679 , 